# 04 – Model v2 and the 2026 time hold-out

Decision D-19 fixed the rules **before** any v2 result was seen:

* v2 is developed on **train** and chosen on **validation** only; primary criterion: validation PR-AUC.
* v2 replaces v1 only if (a) validation PR-AUC ≥ v1 + 0.005, (b) risk never decreases with the planned weight,
  (c) calibration within 2 points in every validation decile, (d) all inputs are known to the coach after the 2nd attempt.
* Threshold rule unchanged: highest precision with recall ≥ 0.70 on validation.
* Final comparison **once** on the 2026 time hold-out (`holdout_2026`, built by notebook 01 with
  `OPL_DATE_MIN=2026-01-01`); reported as is. The 2015–2025 test split is not used here.

v1 is the frozen model card `app/model/model_v1.json` (notebook 02). Section 6 exports v2 for the app (decision D-21):
`app/model/model.json`, `app/model/model-data.js` and `app/tests/reference_cases.json`.

## 0. Setup

In [1]:
import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "docs" / "specs" / "dataset_schema.json").exists():
            return candidate
    raise FileNotFoundError("Start Jupyter from inside the repository.")


ROOT = find_project_root(Path.cwd().resolve())
DATA = ROOT / "data" / "processed" / "deadlift3_attempts.parquet"
HOLDOUT = ROOT / "data" / "processed" / "holdout_2026.parquet"
METADATA = json.loads((ROOT / "data" / "processed" / "metadata.json").read_text(encoding="utf-8"))
V1 = json.loads((ROOT / "app" / "model" / "model_v1.json").read_text(encoding="utf-8"))
REFERENCE = json.loads((ROOT / "app" / "tests" / "reference_cases_v1.json").read_text(encoding="utf-8"))
CARD_V2 = ROOT / "models" / "model_v2_card.json"

EVALUATE_HOLDOUT = True    # evaluated once on 2026-10-25 (D-20); re-runs reproduce it. Was: set True ONCE, after v2 is chosen on validation (D-19)
TARGET_RECALL = 0.70
RANDOM_SEED = 0

df = pd.read_parquet(DATA)
train = df[df["split"] == "train"].reset_index(drop=True)
valid = df[df["split"] == "validation"].reset_index(drop=True)
y_tr, y_va = train["dl3_failed"].to_numpy(float), valid["dl3_failed"].to_numpy(float)
assert V1["dataset"]["opl_commit"] == METADATA["source"]["opl_commit"]
print(f"train {len(train):,} | validation {len(valid):,} | v1 = {V1['model_version']} (threshold {V1['threshold']})")

train 616,069 | validation 131,443 | v1 = monotone-logreg-v1 (threshold 0.3541)


## 1. Features

`v1_features()` repeats the v1 formulas (checked against the reference cases of notebook 02 to 1e-9).
v2 adds four inputs that pass the leakage checklist (spec 05): all are known after the 2nd attempt and the coach can
enter them.

| New feature | Formula | Known because … |
|---|---|---|
| `has_dl1`, `dl1_success` | 1st attempt recorded / good | 1st attempt is done |
| `jump12` | clip(ln(DL2 / DL1), 0, 0.3) | DL1 and DL2 are done |
| `log_prev_meets` | ln(1 + number of earlier competitions) | strictly earlier meets |

The planned weight enters only through `jump` = ln(planned / DL2) and `pr_excess`; both get a monotone
(non-decreasing) constraint, so the predicted risk can never fall when the planned weight rises.

In [2]:
KINKS, PB_CLIP, CATS = V1["constants"]["jump_kinks"], V1["constants"]["pb_gap_clip"], V1["categories"]


def v1_features(frame: pd.DataFrame, planned_kg: np.ndarray) -> pd.DataFrame:
    s = frame["dl2_success"].to_numpy(float); dl2 = frame["dl2_weight_kg"].to_numpy(float)
    bw = frame["bodyweight_kg"].to_numpy(float); pb = frame["prev_best_deadlift_kg"].to_numpy(float)
    has_pb = ~np.isnan(pb); pb_safe = np.where(has_pb, pb, 1.0)
    jump = np.log(np.asarray(planned_kg, float) / dl2)
    gap = np.where(has_pb, np.clip(np.log(dl2 / pb_safe), -PB_CLIP, PB_CLIP), 0.0)
    X = {"dl2_success": s, "rel_strength": np.log(dl2 / bw), "log_bodyweight": np.log(bw),
         "squat_ratio": np.log(frame["best_squat_kg"].to_numpy(float) / dl2),
         "bench_ratio": np.log(frame["best_bench_kg"].to_numpy(float) / dl2),
         "has_prev_best": has_pb.astype(float), "pb_gap": gap, "pb_gap_pos": np.maximum(gap, 0.0)}
    eq, age = frame["equipment"].astype(str).to_numpy(), frame["age_class_group"].astype(str).to_numpy()
    for level in CATS["equipment"]:
        if level != CATS["equipment_baseline"]:
            X[f"equipment_{level}"] = (eq == level).astype(float)
    for level in CATS["age_class_group"]:
        if level != CATS["age_baseline"]:
            X[f"age_{level}"] = (age == level).astype(float)
    X["jump_after_good"], X["jump_after_miss"] = jump * s, jump * (1 - s)
    for kink in KINKS:
        X[f"jump_over_{kink * 100:g}"] = np.maximum(0.0, jump - np.log1p(kink))
    X["pr_excess"] = np.where(has_pb, np.maximum(0.0, np.log(planned_kg / pb_safe)) - np.maximum(0.0, np.log(dl2 / pb_safe)), 0.0)
    return pd.DataFrame(X)


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


V1_COEF = pd.Series(V1["coefficients"])


def predict_v1(frame, planned_kg):
    return sigmoid(V1["intercept"] + v1_features(frame, planned_kg)[V1_COEF.index].to_numpy() @ V1_COEF.to_numpy())


ref = pd.DataFrame([c["context"] for c in REFERENCE["cases"]])
ref["dl2_success"] = ref["dl2_success"].astype(int); ref["prev_best_deadlift_kg"] = ref["prev_best_deadlift_kg"].astype(float)
assert np.max(np.abs(predict_v1(ref, np.array([c["planned_kg"] for c in REFERENCE["cases"]]))
                     - np.array([c["p_fail"] for c in REFERENCE["cases"]]))) < 1e-9


def extra_features(frame: pd.DataFrame) -> pd.DataFrame:
    dl1, dl2 = frame["dl1_weight_kg"].to_numpy(float), frame["dl2_weight_kg"].to_numpy(float)
    has1 = ~np.isnan(dl1)
    return pd.DataFrame({
        "has_dl1": has1.astype(float),
        "dl1_success": np.where(has1, frame["dl1_success"].fillna(1).to_numpy(float), 1.0),
        "jump12": np.where(has1, np.clip(np.log(dl2 / np.where(has1, dl1, 1.0)), 0.0, 0.3), 0.0),
        "log_prev_meets": np.log1p(frame["prev_meets_count"].to_numpy(float)),
    })


def v2_features(frame: pd.DataFrame, planned_kg: np.ndarray, extra: bool = True) -> pd.DataFrame:
    X = v1_features(frame, planned_kg)
    keep = ["dl2_success", "rel_strength", "log_bodyweight", "squat_ratio", "bench_ratio", "has_prev_best", "pb_gap"]
    keep += [c for c in X.columns if c.startswith(("equipment_", "age_"))]
    G = X[keep].copy()
    G["jump"] = np.log(np.asarray(planned_kg, float) / frame["dl2_weight_kg"].to_numpy(float))
    G["pr_excess"] = X["pr_excess"]
    return pd.concat([G, extra_features(frame)], axis=1) if extra else G


MONOTONE = ("jump", "pr_excess")
# Deployed v2 rounds every feature to 9 decimals before the trees (decision D-21): mathematically equal ratios such as
# 125/54.1 and 250/108.2 differ in the last bit, some trees split exactly between them, and numpy and the browser
# compute logarithms differently in that last bit. Rounding makes the prediction identical on every platform.
ROUND_DECIMALS = 9


def round_features(X):
    return np.floor(X * 10.0 ** ROUND_DECIMALS + 0.5) / 10.0 ** ROUND_DECIMALS
p1_va = predict_v1(valid, valid["dl3_weight_kg"].to_numpy())
print("v1 validation PR-AUC", round(average_precision_score(y_va, p1_va), 4))

v1 validation PR-AUC 0.5507


## 2. Candidates (train → validation)

All candidates are fitted on train only. For gradient boosting the number of trees is a hyper-parameter read off
the validation curve (no internal random early-stopping split, which would mix athletes).

In [3]:
def fit_logistic(X, y, monotone_cols, l2=1e-6):
    mean, std = X.mean(axis=0), X.std(axis=0).replace(0, 1.0)
    Z = ((X - mean) / std).to_numpy(); n, k = Z.shape
    def objective(theta):
        z = theta[0] + Z @ theta[1:]
        loss = np.mean(np.logaddexp(0.0, z) - y * z) + l2 / 2 * theta[1:] @ theta[1:]
        r = sigmoid(z) - y
        return loss, np.concatenate([[r.mean()], Z.T @ r / n + l2 * theta[1:]])
    bounds = [(None, None)] + [(0.0, None) if c in monotone_cols else (None, None) for c in X.columns]
    res = minimize(objective, np.zeros(k + 1), jac=True, method="L-BFGS-B", bounds=bounds, options={"maxiter": 5000})
    coef = res.x[1:] / std.to_numpy()
    return res.x[0] - float(np.sum(res.x[1:] * mean.to_numpy() / std.to_numpy())), pd.Series(coef, index=X.columns)


candidates = []
# (1) v1 logistic regression + the new features
Xtr = pd.concat([v1_features(train, train["dl3_weight_kg"].to_numpy()), extra_features(train)], axis=1)
Xva = pd.concat([v1_features(valid, valid["dl3_weight_kg"].to_numpy()), extra_features(valid)], axis=1)
dec = [c for c in Xtr.columns if c.startswith("jump_") or c == "pr_excess"]
b0, c = fit_logistic(Xtr, y_tr, dec)
candidates.append({"candidate": "LR v1 + new features", "pr_auc_val": average_precision_score(y_va, sigmoid(b0 + Xva.to_numpy() @ c.to_numpy()))})

# (2) gradient boosting, monotone in the planned weight
GRID = [(False, 63, 200), (True, 31, 200), (True, 63, 200), (True, 127, 500), (True, 63, 1000)]
gbm = {}
for extra, leaves, msl in GRID:
    Gtr = v2_features(train, train["dl3_weight_kg"].to_numpy(), extra)
    Gva = v2_features(valid, valid["dl3_weight_kg"].to_numpy(), extra)
    model = HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=leaves, min_samples_leaf=msl, max_iter=900,
                                           l2_regularization=1.0, early_stopping=False, random_state=RANDOM_SEED,
                                           monotonic_cst=[1 if col in MONOTONE else 0 for col in Gtr.columns])
    model.fit(Gtr, y_tr)
    scores = {k: average_precision_score(y_va, p[:, 1]) for k, p in enumerate(model.staged_predict_proba(Gva), start=1) if k % 25 == 0}
    best_iter = max(scores, key=scores.get)
    name = f"GBM {'+ new features' if extra else 'v1 features'}, {leaves} leaves, min leaf {msl}, {best_iter} trees"
    candidates.append({"candidate": name, "pr_auc_val": scores[best_iter]})
    gbm[name] = dict(extra=extra, leaves=leaves, msl=msl, iters=best_iter)

table = pd.DataFrame(candidates).sort_values("pr_auc_val", ascending=False).reset_index(drop=True)
table["vs_v1"] = table["pr_auc_val"] - average_precision_score(y_va, p1_va)
table.round(4)

,candidate,pr_auc_val,vs_v1
0,"GBM + new features, 31 leaves, min leaf 200, 8...",0.5693,0.0186
1,"GBM + new features, 63 leaves, min leaf 200, 4...",0.5692,0.0186
2,"GBM + new features, 63 leaves, min leaf 1000, ...",0.5687,0.0180
3,"GBM + new features, 127 leaves, min leaf 500, ...",0.5687,0.0180
4,"GBM v1 features, 63 leaves, min leaf 200, 200 ...",0.5625,0.0118
5,LR v1 + new features,0.5536,0.0030


## 3. Choose v2 on validation and check the D-19 criteria

In [4]:
best = table.iloc[0]["candidate"]
cfg = gbm[best]
FEATURES = list(v2_features(train.head(1), train["dl3_weight_kg"].head(1).to_numpy(), cfg["extra"]).columns)
v2 = HistGradientBoostingClassifier(learning_rate=0.05, max_leaf_nodes=cfg["leaves"], min_samples_leaf=cfg["msl"],
                                    max_iter=cfg["iters"], l2_regularization=1.0, early_stopping=False,
                                    random_state=RANDOM_SEED, monotonic_cst=[1 if c in MONOTONE else 0 for c in FEATURES])
v2.fit(v2_features(train, train["dl3_weight_kg"].to_numpy(), cfg["extra"]), y_tr)


def predict_v2(frame, planned_kg):
    return v2.predict_proba(round_features(v2_features(frame, planned_kg, cfg["extra"])))[:, 1]


def threshold_for_recall(y, score, target=TARGET_RECALL):
    precision, recall, thresholds = precision_recall_curve(y, score)
    ok = recall[:-1] >= target
    return float(thresholds[np.argmax(np.where(ok, precision[:-1], -1))])


def metrics(y, p, threshold):
    flag = p >= threshold; tp = float(np.sum(flag & (y == 1)))
    return {"precision": tp / max(flag.sum(), 1), "recall": tp / max((y == 1).sum(), 1),
            "pr_auc": average_precision_score(y, p), "roc_auc": roc_auc_score(y, p), "n": int(len(y))}


p2_va = predict_v2(valid, valid["dl3_weight_kg"].to_numpy())
T2 = round(threshold_for_recall(y_va, p2_va), 4)

# (b) monotonicity: sweep 2,000 validation situations from DL2 to +25 %
ctx = valid.sample(2000, random_state=7).reset_index(drop=True)
curves = np.stack([predict_v2(ctx, ctx["dl2_weight_kg"].to_numpy() * f) for f in np.linspace(1.0, 1.25, 26)], axis=1)
violations = int((np.diff(curves, axis=1) < -1e-12).any(axis=1).sum())
# (c) calibration in validation deciles
deciles = pd.DataFrame({"p": p2_va, "y": y_va, "bin": pd.qcut(p2_va, 10, duplicates="drop")}).groupby("bin", observed=True).agg(
    predicted=("p", "mean"), observed=("y", "mean"))
max_gap = float((deciles["predicted"] - deciles["observed"]).abs().max())

gain = average_precision_score(y_va, p2_va) - average_precision_score(y_va, p1_va)
checks = {"(a) PR-AUC gain ≥ 0.005": round(gain, 4), "(b) monotonicity violations = 0": violations,
          "(c) max calibration gap ≤ 0.02": round(max_gap, 4), "(d) inputs known to the coach": "yes (section 1)"}
assert gain >= 0.005 and violations == 0 and max_gap <= 0.02, checks
print("v2 =", best, "| threshold", T2); checks

v2 = GBM + new features, 31 leaves, min leaf 200, 825 trees | threshold 0.349


{'(a) PR-AUC gain ≥ 0.005': 0.0184,
 '(b) monotonicity violations = 0': 0,
 '(c) max calibration gap ≤ 0.02': 0.0129,
 '(d) inputs known to the coach': 'yes (section 1)'}

In [5]:
val = pd.DataFrame({"v1": metrics(y_va, p1_va, V1["threshold"]), "v2": metrics(y_va, p2_va, T2)}).T
val.round(4)

,precision,recall,pr_auc,roc_auc,n
v1,0.4831,0.7002,0.5507,0.6650,131443.0
v2,0.4922,0.7000,0.5691,0.6774,131443.0


## 4. Fairness check on validation (spec 07)

Same segments as Table 4 of the report, at v2's validation threshold. `sex` is used only here.

In [6]:
rows = []
for column in ("sex", "age_class_group", "equipment"):
    for value, idx in valid.groupby(column, observed=True).indices.items():
        m = metrics(y_va[idx], p2_va[idx], T2) if y_va[idx].sum() > 0 else {"precision": np.nan, "recall": np.nan}
        rows.append({"segment": column, "value": str(value), "n": len(idx), "failed": int(y_va[idx].sum()),
                     "precision": round(m["precision"], 4), "recall": round(m["recall"], 4)})
segments_v2 = pd.DataFrame(rows)
segments_v2

,segment,value,n,failed,precision,recall
0,sex,M,87031,35714,0.4979,0.7664
1,sex,F,44396,15017,0.4741,0.5423
2,sex,Mx,16,3,0.1667,0.3333
3,age_class_group,Sub-Junior,16791,5918,0.4795,0.5940
4,age_class_group,Junior,32806,13085,0.4955,0.7316
5,age_class_group,Open,52693,21207,0.4984,0.7390
6,age_class_group,Masters,20354,7157,0.4717,0.6279
7,age_class_group,Unknown,8799,3367,0.4988,0.6715
8,equipment,Raw,105004,39347,0.4891,0.6721
9,equipment,Wraps,16757,7008,0.4994,0.7691


## 5. Time hold-out 2026 (run once)

Both frozen models, thresholds from validation, nothing is fitted or chosen here. `EVALUATE_HOLDOUT = True` once;
the results and the date are written to `models/model_v2_card.json`. A re-run with an unchanged v2 reproduces the
same numbers and keeps the first date.

In [7]:
card = {
    "model_version": "gbm-monotone-v2", "chosen": best, "config": cfg, "features": FEATURES, "monotone_features": list(MONOTONE),
    "selection_rule": "D-19: validation PR-AUC >= v1 + 0.005, monotone, calibrated within 2 points, coach-providable inputs",
    "candidates_validation": table.round(4).to_dict(orient="records"), "checks": checks,
    "threshold": T2, "validation": {k: {m: round(v, 4) if isinstance(v, float) else v for m, v in r.items()} for k, r in val.T.to_dict().items()},
    "segments_validation": segments_v2.to_dict(orient="records"), "holdout_2026": None,
}
previous = json.loads(CARD_V2.read_text(encoding="utf-8")) if CARD_V2.exists() else None
if previous and previous.get("holdout_2026"):
    same = previous["chosen"] == best and previous["threshold"] == T2 and previous.get("feature_rounding_decimals") == ROUND_DECIMALS
    card["holdout_2026"] = previous["holdout_2026"] if same else None
    card["holdout_2026_history"] = previous.get("holdout_2026_history", []) + ([] if same else [
        {**previous["holdout_2026"], "model": "v2 without feature rounding, threshold " + str(previous["threshold"])}])
    print("hold-out already evaluated on", previous["holdout_2026"]["evaluated_utc"], "| same v2:", same)
card["feature_rounding_decimals"] = ROUND_DECIMALS

if EVALUATE_HOLDOUT:
    ho = pd.read_parquet(HOLDOUT)
    y_ho = ho["dl3_failed"].to_numpy(float); planned = ho["dl3_weight_kg"].to_numpy()
    p1_ho, p2_ho = predict_v1(ho, planned), predict_v2(ho, planned)
    new_people = (ho["split"] != "train").to_numpy()      # athletes never seen in training
    res = {"v1": metrics(y_ho, p1_ho, V1["threshold"]), "v2": metrics(y_ho, p2_ho, T2),
           "B3 fails if 2nd attempt failed": {"precision": float(np.mean(y_ho[ho["dl2_success"].to_numpy() == 0])),
                                              "recall": float(np.mean(ho["dl2_success"].to_numpy()[y_ho == 1] == 0)),
                                              "pr_auc": average_precision_score(y_ho, 1 - ho["dl2_success"].to_numpy()),
                                              "roc_auc": np.nan, "n": int(len(y_ho))},
           "v1 (athletes not in train)": metrics(y_ho[new_people], p1_ho[new_people], V1["threshold"]),
           "v2 (athletes not in train)": metrics(y_ho[new_people], p2_ho[new_people], T2)}
    evaluated = (card["holdout_2026"] or {}).get("evaluated_utc") or datetime.now(timezone.utc).isoformat(timespec="seconds")
    card["holdout_2026"] = {"evaluated_utc": evaluated, "period": f"{ho['meet_date'].min():%Y-%m-%d} – {ho['meet_date'].max():%Y-%m-%d}",
                            "base_rate": round(float(y_ho.mean()), 4),
                            "results": {k: {m: (round(float(v), 4) if v == v else None) for m, v in r.items()} for k, r in res.items()}}
    display(pd.DataFrame(res).T.round(4))
CARD_V2.parent.mkdir(exist_ok=True)
CARD_V2.write_text(json.dumps(card, indent=2, ensure_ascii=False, default=float) + "\n", encoding="utf-8")
print("wrote", CARD_V2.relative_to(ROOT))

hold-out already evaluated on 2026-10-25T19:02:49+00:00 | same v2: False


,precision,recall,pr_auc,roc_auc,n
v1,0.4926,0.7159,0.5622,0.6645,81900.0
v2,0.4996,0.7190,0.5819,0.6766,81900.0
B3 fails if 2nd attempt failed,0.6187,0.1720,0.4370,NaN,81900.0
v1 (athletes not in train),0.4895,0.7186,0.5553,0.6646,24500.0
v2 (athletes not in train),0.4947,0.7190,0.5764,0.6762,24500.0


wrote models/model_v2_card.json


## 6. Export v2 for the app (decision D-21)

The app evaluates the trees itself (no server): each tree is stored as arrays – `f` split feature (−1 = leaf),
`t` threshold (go left if value ≤ t), `l`/`r` children, `v` leaf value. Features are first rounded to 9 decimals
(`floor(x · 1e9 + 0.5) / 1e9`, see section 1). Risk = sigmoid(baseline + sum of the leaf values).
A pure-Python re-implementation of exactly this rule must reproduce `predict_proba` before anything is written, and the
JavaScript tests must reproduce the reference probabilities to 1e-9.

In [8]:
import hashlib
SCHEMA = json.loads((ROOT / "docs" / "specs" / "dataset_schema.json").read_text(encoding="utf-8"))
APP_MODEL_DIR, APP_TEST_DIR = ROOT / "app" / "model", ROOT / "app" / "tests"

trees = []
for (predictor,) in v2._predictors:
    nd = predictor.nodes
    assert not nd["is_categorical"].any()
    leaf = nd["is_leaf"].astype(bool)
    trees.append({"f": np.where(leaf, -1, nd["feature_idx"]).astype(int).tolist(),
                  "t": np.where(leaf, 0.0, nd["num_threshold"]).astype(float).tolist(),
                  "l": nd["left"].astype(int).tolist(), "r": nd["right"].astype(int).tolist(),
                  "v": np.where(leaf, nd["value"], 0.0).astype(float).tolist()})
BASELINE = float(np.ravel(v2._baseline_prediction)[0])


def predict_from_trees(X: np.ndarray) -> np.ndarray:
    X = round_features(np.asarray(X, float))
    raw = np.full(len(X), BASELINE)
    for tree in trees:
        f, t, l, r, v = (np.asarray(tree[k]) for k in ("f", "t", "l", "r", "v"))
        node = np.zeros(len(X), dtype=int)
        for _ in range(64):
            inner = f[node] >= 0
            if not inner.any():
                break
            go_left = X[np.arange(len(X)), np.maximum(f[node], 0)] <= t[node]
            node = np.where(inner, np.where(go_left, l[node], r[node]), node)
        raw += v[node]
    return sigmoid(raw)


Xcheck = v2_features(valid.head(5000), valid["dl3_weight_kg"].head(5000).to_numpy(), cfg["extra"])
assert np.max(np.abs(predict_from_trees(Xcheck.to_numpy()) - predict_v2(valid.head(5000), valid["dl3_weight_kg"].head(5000).to_numpy()))) < 1e-12
print(len(trees), "trees,", sum(len(tr["f"]) for tr in trees), "nodes; tree export reproduces predict_proba")

825 trees, 50325 nodes; tree export reproduces predict_proba


In [9]:
# Everything the app shows about the model, in the same layout as the v1 card (spec 08)
schema_cols = {c["name"]: c for c in SCHEMA["columns"]}
INPUTS = ["dl1_weight_kg", "dl2_weight_kg", "bodyweight_kg", "best_squat_kg", "best_bench_kg", "prev_best_deadlift_kg", "prev_meets_count"]
ranges = METADATA["feature_ranges_train"]
input_bounds = {n: {"min": schema_cols[n]["min"], "max": schema_cols[n]["max"]} for n in INPUTS}
input_typical = {n: {"p01": round(ranges[n]["p01"], 2), "p99": round(ranges[n]["p99"], 2)} for n in INPUTS}
G_tr = v2_features(train, train["dl3_weight_kg"].to_numpy(), cfg["extra"])
profile_typical = {}
for n in ("rel_strength", "squat_ratio", "bench_ratio", "pb_gap"):
    vals = G_tr[n] if n != "pb_gap" else G_tr.loc[G_tr["has_prev_best"] == 1, n]
    profile_typical[n] = {"p01": round(float(vals.quantile(0.01)), 4), "p99": round(float(vals.quantile(0.99)), 4)}

DOCS = {"dl2_success": "1 if the 2nd attempt was good", "rel_strength": "ln(DL2 / bodyweight)", "log_bodyweight": "ln(bodyweight)",
        "squat_ratio": "ln(best squat / DL2)", "bench_ratio": "ln(best bench / DL2)", "has_prev_best": "1 if a previous competition best is given",
        "pb_gap": f"clip(ln(DL2 / previous best), -{PB_CLIP}, {PB_CLIP}), 0 without previous best",
        "jump": "ln(planned / DL2)", "pr_excess": "max(0, ln(planned / previous best)) - max(0, ln(DL2 / previous best))",
        "has_dl1": "1 if the 1st attempt is given", "dl1_success": "1 if the 1st attempt was good",
        "jump12": "clip(ln(DL2 / DL1), 0, 0.3)", "log_prev_meets": "ln(1 + number of earlier competitions)"}
def doc(n):
    if n.startswith("equipment_"): return f"1 if equipment is {n[10:]} (baseline {CATS['equipment_baseline']})"
    if n.startswith("age_"): return f"1 if age group is {n[4:]} (baseline {CATS['age_baseline']})"
    return DOCS[n]

ho = card["holdout_2026"]
y_flag = p2_va >= T2
jump_va = valid["dl3_jump_pct"].to_numpy()
b2_cut = threshold_for_recall(y_va, jump_va)
def bm(flag, score):
    tp = float(np.sum(flag & (y_va == 1)))
    return {"precision": round(tp / max(flag.sum(), 1), 4), "recall": round(tp / (y_va == 1).sum(), 4),
            "pr_auc": round(average_precision_score(y_va, score), 4)}
baselines = {"B1 always succeeds": {"precision": None, "recall": 0.0, "pr_auc": round(float(y_va.mean()), 4)},
             f"B2 fails if jump ≥ {b2_cut:g} %": bm(jump_va >= b2_cut, jump_va),
             "B3 fails if 2nd attempt failed": bm(valid["dl2_success"].to_numpy() == 0, 1 - valid["dl2_success"].to_numpy()),
             "Model v1 (logistic regression)": bm(p1_va >= V1["threshold"], p1_va),
             "Model v2 (gradient boosting, in use)": bm(y_flag, p2_va)}
calib = deciles.reset_index(drop=True).assign(n=pd.Series(pd.qcut(p2_va, 10, duplicates="drop")).value_counts(sort=False).to_numpy())
overall_recall = metrics(y_va, p2_va, T2)["recall"]
seg = segments_v2.assign(reliable=lambda d: d["failed"] >= 100,
                         recall_gap_pp=lambda d: ((d["recall"] - overall_recall) * 100).round(1))
seg["flag"] = np.where(~seg["reliable"], "too small", np.where(seg["recall_gap_pp"].abs() > 10, "check", ""))
r4 = lambda d: {k: (round(float(v), 4) if isinstance(v, (float, np.floating)) else v) for k, v in d.items()}

app_card = {
    "model_version": "gbm-monotone-v2", "model_kind": "gbm",
    "display_name": f"Gradient boosting ({len(trees)} small decision trees)",
    "model_type": f"Gradient boosting, {len(trees)} trees, max. {cfg['leaves']} leaves, monotone (non-decreasing) in the planned 3rd-attempt weight",
    "target": "probability that the 3rd deadlift attempt fails (dl3_failed = 1)",
    "dataset": {"snapshot_date": METADATA["source"]["snapshot_date"], "opl_commit": METADATA["source"]["opl_commit"],
                "schema_version": SCHEMA["schema_version"], "period": METADATA["summary"]["period"],
                "train_records": int(len(train)), "train_athletes": int(train["athlete_id"].nunique()), "validation_records": int(len(valid))},
    "features": [{"name": n, "formula": doc(n), "grows_with_planned_weight": n in MONOTONE} for n in FEATURES],
    "baseline": BASELINE, "trees": trees, "feature_rounding_decimals": ROUND_DECIMALS,
    "constants": {"pb_gap_clip": PB_CLIP, "jump12_clip": [0.0, 0.3]},
    "categories": CATS,
    "threshold": T2, "threshold_rule": f"highest precision with recall >= {TARGET_RECALL} on the validation split",
    "uncertainty_band": V1["uncertainty_band"], "max_jump_pct": V1["max_jump_pct"],
    "input_bounds": input_bounds, "input_typical": input_typical, "profile_typical": profile_typical,
    "metrics": {"validation": {**r4(metrics(y_va, p2_va, T2)), "threshold": T2, "base_rate": round(float(y_va.mean()), 4)},
                "test": None,
                "holdout_2026": None if ho is None else {**ho["results"]["v2"], "evaluated_utc": ho["evaluated_utc"],
                                                         "period": ho["period"], "base_rate": ho["base_rate"]}},
    "comparison_v1": {"validation": r4(metrics(y_va, p1_va, V1["threshold"])),
                      "holdout_2026": None if ho is None else ho["results"]["v1"]},
    "baselines_validation": baselines,
    "monotonicity_check_validation": {"situations_where_more_weight_looks_safer": round(violations / len(ctx), 4)},
    "calibration_validation": calib.round(4).to_dict(orient="records"),
    "segments_validation": seg.to_dict(orient="records"),
    "selection": {"rule": card["selection_rule"], "candidates_validation": card["candidates_validation"], "checks": checks},
    "limitations": [
        "Learned from competition results: coaches already pick smaller jumps after a slow 2nd attempt, so the data cannot "
        "show how much safer a small jump is for a given athlete. The risk is constrained never to fall when the weight rises.",
        "Day form, warm-ups, bar speed and weight cutting are not in the data. Treat the output as one input to the coach's decision.",
        "Federations judge differently, and only the last declared 3rd-attempt weight is recorded.",
        "Small groups (Mx, Multi-ply, Unlimited, very young or old lifters) have few cases; predictions there are less reliable.",
        "A tree ensemble is harder to explain than v1's logistic regression; v1 stays in the repository as the tested baseline.",
    ],
    "attribution": METADATA["source"]["attribution"],
}
trees_hash = hashlib.sha256(json.dumps(trees).encode()).hexdigest()[:16]
old = json.loads((APP_MODEL_DIR / "model.json").read_text(encoding="utf-8")) if (APP_MODEL_DIR / "model.json").exists() else {}
same_model = old.get("model_version") == app_card["model_version"] and old.get("trees_sha256") == trees_hash
app_card["trees_sha256"] = trees_hash
app_card["created_utc"] = old["created_utc"] if same_model else datetime.now(timezone.utc).isoformat(timespec="seconds")

payload = json.dumps(app_card, ensure_ascii=False, separators=(",", ":"), default=float)
(APP_MODEL_DIR / "model.json").write_text(payload + "\n", encoding="utf-8")
(APP_MODEL_DIR / "model-data.js").write_text("// Generated by notebooks/04_model_v2.ipynb – do not edit by hand.\n"
                                             f"window.DEADLIFT_MODEL = {payload};\n", encoding="utf-8")
print(f"wrote app/model/model.json ({len(payload) / 1e6:.1f} MB) and model-data.js")

wrote app/model/model.json (1.6 MB) and model-data.js


In [10]:
# Reference cases: the JavaScript must reproduce these probabilities to 1e-9 (app/tests/model.test.js)
cases = valid[valid["dl1_weight_kg"].notna()].sample(40, random_state=7).reset_index(drop=True)
factor = np.random.default_rng(7).uniform(1.0, 1.2, len(cases))
dl2c = cases["dl2_weight_kg"].to_numpy()
planned = np.maximum(np.round(dl2c * factor, 1), dl2c)
planned = np.where(cases["dl2_success"].to_numpy() == 1, np.maximum(planned, dl2c + 0.5), planned)
p_cases = predict_from_trees(v2_features(cases, planned, cfg["extra"]).to_numpy())
reference = [{"context": {
                 "dl1_weight_kg": float(r.dl1_weight_kg), "dl1_success": bool(r.dl1_success),
                 "dl2_weight_kg": float(r.dl2_weight_kg), "dl2_success": bool(r.dl2_success),
                 "bodyweight_kg": float(r.bodyweight_kg), "age_class_group": str(r.age_class_group), "equipment": str(r.equipment),
                 "best_squat_kg": float(r.best_squat_kg), "best_bench_kg": float(r.best_bench_kg),
                 "prev_best_deadlift_kg": None if pd.isna(r.prev_best_deadlift_kg) else float(r.prev_best_deadlift_kg),
                 "prev_meets_count": int(r.prev_meets_count)},
              "planned_kg": float(planned[i]), "p_fail": float(p_cases[i])} for i, r in cases.iterrows()]
(APP_TEST_DIR / "reference_cases.json").write_text(json.dumps({"model_version": app_card["model_version"], "cases": reference}, indent=1) + "\n")
print(len(reference), "reference cases written")

40 reference cases written
